# 05A — Modeling Table Assembly and Feature-Provenance Audit

**Build ID: `05A_V1_MODELING_TABLE_ASSEMBLY_AND_FEATURE_MANIFEST`**

This notebook assembles the frozen **conditional line-hold modeling universe**
after completion of the dynamic approach, HRRR, terrain, and five-vintage
LANDFIRE feature pipelines.

## Inputs

1. 04A V3 exact dynamic fire-approach reconstruction
2. 04B1 V2 cleaned operational HRRR forecast features
3. 04C1 V1 USGS 3DEP terrain features
4. 04D1J V1 five-vintage LANDFIRE features
5. 03N frozen cohort, used only as the authoritative target/metadata cross-check

## Expected operational universe

**304,143 units**

This notebook does **not** train a model.

It first establishes a single scientifically auditable modeling table and a
column manifest that identifies:

- target;
- incident split key;
- identifiers;
- source provenance;
- physical candidate predictors;
- QA / availability fields;
- fields that require explicit review before inclusion.

The goal is to prevent silent leakage or accidental use of identifiers before
incident-held-out modeling begins.

## Assembly principles

- `unit_uid` is the only row-join key.
- Every production source must have unique `unit_uid`.
- HRRR, terrain, and LANDFIRE must cover exactly the same 304,143-unit set.
- 04A may contain a larger upstream timing cohort, but its rows for the
  operational universe must match the 304,143-unit set exactly.
- `hold_label` is the conditional-hold target.
- `_irwin_norm` is the fire/incident split key and is never a predictor.
- `_engagement_norm` is outcome-derived and is never a predictor.
- `IncidentName`, raw coordinates, timestamps, block IDs, runtime fields, and
  source identifiers are not default predictors.
- LANDFIRE tier information comes from the frozen V5 feature dictionary.
- Missing values are preserved. No feature imputation occurs in 05A.
- No random row split is created here.

## 0. Configuration

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = (
    "05A_V1_MODELING_TABLE_ASSEMBLY_AND_FEATURE_MANIFEST"
)

from pathlib import Path
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path(
    os.environ.get("FIRELINE_PROJECT_ROOT", ".")
)

FROZEN_COHORT_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "fireline_corrected_local_250m_10m" /
    "frozen_cohort" /
    "fireline_ml_dynamic_timing_cohort_frozen_v1.parquet"
)

APPROACH_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "fireline_extended_exact_approach_v3" /
    "extended_exact_approach_features_v3.parquet"
)

HRRR_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "hrrr_production_v2" /
    "hrrr_operational_forecast_features_v2_clean.parquet"
)

TERRAIN_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "terrain_3dep_production_v1" /
    "terrain_3dep_features_v1.parquet"
)

LANDFIRE_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "landfire_five_vintage_consolidated_v1" /
    "landfire_features_five_vintage_consolidated_v1.parquet"
)

LANDFIRE_DICTIONARY_PATH = (
    PROJECT_ROOT / "outputs" /
    "landfire_feature_architecture_v5" /
    "LANDFIRE_FEATURE_DICTIONARY_V5.csv"
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "outputs" /
    "modeling_table_assembly_v1"
)

PROCESSED_ROOT = (
    PROJECT_ROOT / "data" / "processed" /
    "modeling_table_assembly_v1"
)

MASTER_PATH = (
    PROCESSED_ROOT /
    "fireline_hold_modeling_master_v1.parquet"
)

for p in [
    OUTPUT_ROOT,
    PROCESSED_ROOT,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

for p in [
    FROZEN_COHORT_PATH,
    APPROACH_PATH,
    HRRR_PATH,
    TERRAIN_PATH,
    LANDFIRE_PATH,
    LANDFIRE_DICTIONARY_PATH,
]:
    assert p.exists(), p

EXPECTED_OPERATIONAL_UNITS = 304_143

# Frozen QA counts already established upstream.
EXPECTED_APPROACH_DIRECTION_RELIABLE = 291_228
EXPECTED_APPROACH_DIRECTION_UNRELIABLE = 12_915

# Full equality audit for shared scientific-contract columns.
SHARED_CONTRACT_COLUMNS = [
    "_irwin_norm",
    "IncidentName",
    "Year",
    "_engagement_norm",
    "hold_label",
    "prediction_anchor_utc",
    "longitude",
    "latitude",
    "approach_azimuth_deg",
    "approach_direction_reliable",
    "approach_vs_line_angle_deg",
]

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)
print("Expected operational units:", EXPECTED_OPERATIONAL_UNITS)
print("Master output:", MASTER_PATH)

## 1. Imports

In [ ]:
required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
    "duckdb": "duckdb",
}

for module, package in required.items():
    if importlib.util.find_spec(
        module
    ) is None:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                package,
            ]
        )

from datetime import datetime, timezone
import json
import math
import re
import time

import numpy as np
import pandas as pd

import pyarrow.parquet as pq
import duckdb

pd.set_option(
    "display.max_columns",
    500,
)
pd.set_option(
    "display.max_rows",
    500,
)
pd.set_option(
    "display.width",
    420,
)

## 2. Source schema inventory

In [ ]:
SOURCE_PATHS = {
    "frozen_cohort": FROZEN_COHORT_PATH,
    "approach": APPROACH_PATH,
    "hrrr": HRRR_PATH,
    "terrain": TERRAIN_PATH,
    "landfire": LANDFIRE_PATH,
}

schema_rows = []
source_columns = {}

for source, path in SOURCE_PATHS.items():

    meta = pq.read_metadata(
        path
    )

    columns = list(
        meta.schema.names
    )

    source_columns[
        source
    ] = columns

    schema_rows.append({
        "source": source,
        "path": str(
            path
        ),
        "rows": int(
            meta.num_rows
        ),
        "columns": len(
            columns
        ),
        "has_unit_uid": (
            "unit_uid"
            in columns
        ),
        "has_hold_label": (
            "hold_label"
            in columns
        ),
        "has_irwin": (
            "_irwin_norm"
            in columns
        ),
    })


source_schema = pd.DataFrame(
    schema_rows
)

display(
    source_schema
)

assert source_schema[
    "has_unit_uid"
].all()

assert int(
    source_schema.loc[
        source_schema[
            "source"
        ] == "hrrr",
        "rows",
    ].iloc[
        0
    ]
) == EXPECTED_OPERATIONAL_UNITS

assert int(
    source_schema.loc[
        source_schema[
            "source"
        ] == "terrain",
        "rows",
    ].iloc[
        0
    ]
) == EXPECTED_OPERATIONAL_UNITS

assert int(
    source_schema.loc[
        source_schema[
            "source"
        ] == "landfire",
        "rows",
    ].iloc[
        0
    ]
) == EXPECTED_OPERATIONAL_UNITS

source_schema.to_csv(
    OUTPUT_ROOT /
    "SOURCE_SCHEMA_INVENTORY_V1.csv",
    index=False,
)

print(
    "SOURCE SCHEMA INVENTORY: PASS"
)

## 3. Exact operational-unit set audit

In [ ]:
def read_unit_ids(
    path,
):
    x = pd.read_parquet(
        path,
        columns=[
            "unit_uid",
        ],
    )

    if not x[
        "unit_uid"
    ].is_unique:
        raise RuntimeError(
            f"Duplicate unit_uid in {path}"
        )

    return set(
        x[
            "unit_uid"
        ]
    )


hrrr_ids = read_unit_ids(
    HRRR_PATH
)

terrain_ids = read_unit_ids(
    TERRAIN_PATH
)

landfire_ids = read_unit_ids(
    LANDFIRE_PATH
)

approach_ids_all = read_unit_ids(
    APPROACH_PATH
)

frozen_ids_all = read_unit_ids(
    FROZEN_COHORT_PATH
)

assert len(
    hrrr_ids
) == EXPECTED_OPERATIONAL_UNITS

id_audit_rows = []

for source, ids in [
    (
        "hrrr",
        hrrr_ids,
    ),
    (
        "terrain",
        terrain_ids,
    ),
    (
        "landfire",
        landfire_ids,
    ),
    (
        "approach_all",
        approach_ids_all,
    ),
    (
        "frozen_cohort_all",
        frozen_ids_all,
    ),
]:

    id_audit_rows.append({
        "source": source,
        "unique_ids": len(
            ids
        ),
        "missing_vs_hrrr": len(
            hrrr_ids
            - ids
        ),
        "extra_vs_hrrr": len(
            ids
            - hrrr_ids
        ),
        "contains_all_operational_ids": (
            len(
                hrrr_ids
                - ids
            )
            == 0
        ),
        "exact_operational_id_set": (
            ids
            == hrrr_ids
        ),
    })


id_audit = pd.DataFrame(
    id_audit_rows
)

display(
    id_audit
)

assert terrain_ids == hrrr_ids
assert landfire_ids == hrrr_ids
assert hrrr_ids.issubset(
    approach_ids_all
)
assert hrrr_ids.issubset(
    frozen_ids_all
)

id_audit.to_csv(
    OUTPUT_ROOT /
    "OPERATIONAL_UNIT_SET_AUDIT_V1.csv",
    index=False,
)

print(
    "EXACT OPERATIONAL UNIT SET: PASS"
)

## 4. Authoritative target and metadata consistency

Only a small set of contract columns is loaded into memory.

The authoritative target check is:

`03N frozen cohort hold_label == 04A hold_label == HRRR hold_label == terrain hold_label`

where those columns exist.

The notebook also checks incident identity, year, engagement label, approach
direction, and other duplicated contract fields whenever the same field is
present in multiple source products.

In [ ]:
def available_columns(
    source,
    requested,
):
    cols = source_columns[
        source
    ]

    return [
        c
        for c in requested
        if c in cols
    ]


contract_load_cols = {
    source: [
        "unit_uid",
        *available_columns(
            source,
            SHARED_CONTRACT_COLUMNS,
        ),
    ]
    for source in SOURCE_PATHS
}


contract_frames = {}

for source, path in SOURCE_PATHS.items():

    cols = contract_load_cols[
        source
    ]

    frame = pd.read_parquet(
        path,
        columns=cols,
    )

    frame = frame[
        frame[
            "unit_uid"
        ].isin(
            hrrr_ids
        )
    ].copy()

    assert len(
        frame
    ) == EXPECTED_OPERATIONAL_UNITS

    assert frame[
        "unit_uid"
    ].is_unique

    contract_frames[
        source
    ] = frame.set_index(
        "unit_uid"
    ).sort_index()


def series_equal_with_na(
    a,
    b,
):
    a = a.copy()
    b = b.copy()

    # Datetime/string representation can differ while retaining equal timestamps.
    if (
        "datetime" in str(
            a.dtype
        ).lower()
        or "datetime" in str(
            b.dtype
        ).lower()
    ):
        aa = pd.to_datetime(
            a,
            utc=True,
            errors="coerce",
        )

        bb = pd.to_datetime(
            b,
            utc=True,
            errors="coerce",
        )

        return (
            (
                aa == bb
            )
            | (
                aa.isna()
                & bb.isna()
            )
        )

    # -------------------------------------------------------------
    # Numeric comparison.
    #
    # Treat the columns as numeric only if every nonmissing original
    # value successfully converts to numeric.
    # -------------------------------------------------------------
    aa_num = pd.to_numeric(
        a,
        errors="coerce",
    )

    bb_num = pd.to_numeric(
        b,
        errors="coerce",
    )

    numeric_support = (
        (
            a.notna()
            & aa_num.notna()
        ).sum()
        + (
            b.notna()
            & bb_num.notna()
        ).sum()
    )

    original_support = (
        a.notna().sum()
        + b.notna().sum()
    )

    if (
        original_support > 0
        and numeric_support
        == original_support
    ):
        eq = np.isclose(
            aa_num.to_numpy(
                dtype=float
            ),
            bb_num.to_numpy(
                dtype=float
            ),
            equal_nan=True,
            rtol=1e-10,
            atol=1e-10,
        )

        return pd.Series(
            eq,
            index=a.index,
        )

    # -------------------------------------------------------------
    # General string/object comparison.
    # Preserve missing == missing semantics.
    # -------------------------------------------------------------
    aa = a.astype(
        "string"
    )

    bb = b.astype(
        "string"
    )

    return (
        (
            aa == bb
        )
        | (
            aa.isna()
            & bb.isna()
        )
    )


contract_audit_rows = []

sources = list(
    contract_frames
)

for column in SHARED_CONTRACT_COLUMNS:

    holders = [
        source
        for source in sources
        if column
        in contract_frames[
            source
        ].columns
    ]

    if len(
        holders
    ) < 2:
        continue

    reference_source = (
        "frozen_cohort"
        if "frozen_cohort"
        in holders
        else holders[
            0
        ]
    )

    reference = contract_frames[
        reference_source
    ][
        column
    ]

    for source in holders:

        if source == reference_source:
            continue

        comparison = contract_frames[
            source
        ][
            column
        ]

        eq = series_equal_with_na(
            reference,
            comparison,
        )

        mismatches = int(
            (
                ~eq
            ).sum()
        )

        contract_audit_rows.append({
            "column": column,
            "reference_source": reference_source,
            "comparison_source": source,
            "rows_compared": len(
                eq
            ),
            "mismatches": mismatches,
            "exact_contract_match": (
                mismatches
                == 0
            ),
        })


contract_audit = pd.DataFrame(
    contract_audit_rows
)

display(
    contract_audit
)

if len(
    contract_audit
):
    assert contract_audit[
        "exact_contract_match"
    ].all()

# Explicit target presence and consistency.
for source in [
    "frozen_cohort",
    "approach",
    "hrrr",
    "terrain",
]:
    assert "hold_label" in contract_frames[
        source
    ].columns

target = contract_frames[
    "frozen_cohort"
][
    "hold_label"
]

assert target.notna().all()

target_counts = (
    target.value_counts(
        dropna=False
    )
    .rename_axis(
        "hold_label"
    )
    .reset_index(
        name="units"
    )
)

target_counts[
    "pct"
] = (
    target_counts[
        "units"
    ]
    / len(
        target
    )
    * 100.0
)

display(
    target_counts
)

contract_audit.to_csv(
    OUTPUT_ROOT /
    "SHARED_CONTRACT_COLUMN_AUDIT_V1.csv",
    index=False,
)

target_counts.to_csv(
    OUTPUT_ROOT /
    "TARGET_DISTRIBUTION_V1.csv",
    index=False,
)

print(
    "TARGET / METADATA CONSISTENCY: PASS"
)

## 5. 04A operational-anchor and direction audit

In [ ]:
approach_audit_cols = [
    "unit_uid",
    "operational_anchor_clean",
    "approach_direction_reliable",
    "approach_azimuth_deg",
]

approach_audit_cols = [
    c
    for c in approach_audit_cols
    if c in source_columns[
        "approach"
    ]
]

approach_operational_qa = pd.read_parquet(
    APPROACH_PATH,
    columns=approach_audit_cols,
)

approach_operational_qa = (
    approach_operational_qa[
        approach_operational_qa[
            "unit_uid"
        ].isin(
            hrrr_ids
        )
    ]
    .copy()
)

assert len(
    approach_operational_qa
) == EXPECTED_OPERATIONAL_UNITS

if (
    "operational_anchor_clean"
    in approach_operational_qa.columns
):
    assert approach_operational_qa[
        "operational_anchor_clean"
    ].fillna(
        False
    ).all()

direction_reliable = (
    approach_operational_qa[
        "approach_direction_reliable"
    ].fillna(
        False
    )
)

direction_finite = np.isfinite(
    pd.to_numeric(
        approach_operational_qa[
            "approach_azimuth_deg"
        ],
        errors="coerce",
    )
)

direction_valid = (
    direction_reliable
    & direction_finite
)

direction_qa = pd.DataFrame({
    "metric": [
        "operational_units",
        "approach_direction_valid",
        "approach_direction_invalid",
    ],
    "value": [
        len(
            approach_operational_qa
        ),
        int(
            direction_valid.sum()
        ),
        int(
            (
                ~direction_valid
            ).sum()
        ),
    ],
})

display(
    direction_qa
)

assert int(
    direction_valid.sum()
) == EXPECTED_APPROACH_DIRECTION_RELIABLE

assert int(
    (
        ~direction_valid
    ).sum()
) == EXPECTED_APPROACH_DIRECTION_UNRELIABLE

direction_qa.to_csv(
    OUTPUT_ROOT /
    "APPROACH_DIRECTION_AUDIT_V1.csv",
    index=False,
)

print(
    "04A OPERATIONAL DIRECTION CONTRACT: PASS"
)

## 6. Build column-source plan

Assembly priority is intentional:

1. **core** identity / target / split metadata;
2. **04A approach** columns;
3. HRRR columns not already supplied by core/04A;
4. terrain columns not already supplied upstream;
5. exactly the **605 frozen V5 LANDFIRE fields**.

This avoids pandas suffixes and keeps one source of truth for duplicated
metadata.

All 605 LANDFIRE fields come directly from the frozen feature dictionary.

In [ ]:
landfire_dictionary = pd.read_csv(
    LANDFIRE_DICTIONARY_PATH
)

assert landfire_dictionary[
    "feature"
].is_unique

LANDFIRE_FEATURES = list(
    landfire_dictionary[
        "feature"
    ].astype(str)
)

assert len(
    LANDFIRE_FEATURES
) == 605

assert set(
    LANDFIRE_FEATURES
).issubset(
    set(
        source_columns[
            "landfire"
        ]
    )
)

CORE_COLUMNS_REQUESTED = [
    "unit_uid",
    "_irwin_norm",
    "IncidentName",
    "Year",
    "_engagement_norm",
    "hold_label",
    "prediction_anchor_utc",
    "longitude",
    "latitude",
]

CORE_COLUMNS = [
    c
    for c in CORE_COLUMNS_REQUESTED
    if c in source_columns[
        "hrrr"
    ]
]

required_core = [
    "unit_uid",
    "_irwin_norm",
    "IncidentName",
    "Year",
    "_engagement_norm",
    "hold_label",
]

missing_core = [
    c
    for c in required_core
    if c not in CORE_COLUMNS
]

if missing_core:
    raise RuntimeError(
        f"HRRR clean table missing core columns: {missing_core}"
    )


APPROACH_EXCLUDE_FROM_ASSEMBLY = {
    *CORE_COLUMNS,
}

APPROACH_COLUMNS = [
    c
    for c in source_columns[
        "approach"
    ]
    if c not in APPROACH_EXCLUDE_FROM_ASSEMBLY
]


already_selected = set(
    CORE_COLUMNS
    + APPROACH_COLUMNS
)

HRRR_COLUMNS = [
    c
    for c in source_columns[
        "hrrr"
    ]
    if c not in already_selected
]


already_selected.update(
    HRRR_COLUMNS
)

TERRAIN_COLUMNS = [
    c
    for c in source_columns[
        "terrain"
    ]
    if c not in already_selected
]


already_selected.update(
    TERRAIN_COLUMNS
)

LANDFIRE_COLUMNS = [
    c
    for c in LANDFIRE_FEATURES
    if c not in already_selected
]


FINAL_COLUMNS = (
    CORE_COLUMNS
    + APPROACH_COLUMNS
    + HRRR_COLUMNS
    + TERRAIN_COLUMNS
    + LANDFIRE_COLUMNS
)

assert len(
    FINAL_COLUMNS
) == len(
    set(
        FINAL_COLUMNS
    )
)

source_plan_rows = []

for source, columns in [
    (
        "core_hrrr",
        CORE_COLUMNS,
    ),
    (
        "approach",
        APPROACH_COLUMNS,
    ),
    (
        "hrrr",
        HRRR_COLUMNS,
    ),
    (
        "terrain",
        TERRAIN_COLUMNS,
    ),
    (
        "landfire",
        LANDFIRE_COLUMNS,
    ),
]:

    source_plan_rows.append({
        "source": source,
        "selected_columns": len(
            columns
        ),
    })


source_plan = pd.DataFrame(
    source_plan_rows
)

display(
    source_plan
)

print(
    "Final assembled columns:",
    len(
        FINAL_COLUMNS
    ),
)

source_plan.to_csv(
    OUTPUT_ROOT /
    "ASSEMBLY_SOURCE_COLUMN_COUNTS_V1.csv",
    index=False,
)

pd.DataFrame({
    "column": FINAL_COLUMNS,
    "position": np.arange(
        len(
            FINAL_COLUMNS
        )
    ),
}).to_csv(
    OUTPUT_ROOT /
    "ASSEMBLY_CANONICAL_COLUMN_ORDER_V1.csv",
    index=False,
)

## 7. Build feature/provenance manifest

This is a **conservative first-pass role assignment**. It is not yet the final
model specification.

Anything marked `feature_candidate` can be considered in the next modeling
architecture notebook. Anything marked `id`, `target`, `target_derived`,
`split_key`, `metadata`, `qa`, or `provenance` is excluded from the default
predictor set.

The next step will review candidate predictors before fitting.

In [ ]:
source_of_column = {}

for c in CORE_COLUMNS:
    source_of_column[
        c
    ] = "core"

for c in APPROACH_COLUMNS:
    source_of_column[
        c
    ] = "approach"

for c in HRRR_COLUMNS:
    source_of_column[
        c
    ] = "hrrr"

for c in TERRAIN_COLUMNS:
    source_of_column[
        c
    ] = "terrain"

for c in LANDFIRE_COLUMNS:
    source_of_column[
        c
    ] = "landfire"


landfire_tier_map = dict(
    zip(
        landfire_dictionary[
            "feature"
        ].astype(str),
        landfire_dictionary[
            "tier"
        ].astype(str),
    )
)

landfire_product_map = dict(
    zip(
        landfire_dictionary[
            "feature"
        ].astype(str),
        landfire_dictionary[
            "product"
        ].astype(str),
    )
)


TARGET_COLUMNS = {
    "hold_label",
}

TARGET_DERIVED_COLUMNS = {
    "_engagement_norm",
}

ID_COLUMNS = {
    "unit_uid",
    "IncidentName",
}

SPLIT_COLUMNS = {
    "_irwin_norm",
}

METADATA_COLUMNS = {
    "Year",
    "prediction_anchor_utc",
    "longitude",
    "latitude",
}

# Explicitly conservative exclusion tokens.
QA_TOKENS = (
    "valid_time_error",
    "runtime",
    "source_tile",
    "source_status",
    "available",
    "availability",
    "fallback",
    "store_lead",
    "patch_valid",
    "direction_valid",
    "direction_reliable",
    "operational_anchor_clean",
    "dynamic_feature_ready",
    "any_inside",
)

PROVENANCE_TOKENS = (
    "selected_init",
    "primary_extended_init",
    "_source",
    "_irwin_04a",
    "block_id",
)

POTENTIAL_LEAKAGE_TOKENS = (
    "future",
    "outcome",
    "label",
    "challenge_end",
    "post_contact",
    "after_contact",
)


def classify_column(
    column,
):
    lc = column.lower()

    if column in TARGET_COLUMNS:
        return (
            "target",
            False,
            "conditional hold target",
        )

    if column in TARGET_DERIVED_COLUMNS:
        return (
            "target_derived",
            False,
            "outcome-derived category; never a predictor",
        )

    if column in ID_COLUMNS:
        return (
            "id",
            False,
            "identifier; never a predictor",
        )

    if column in SPLIT_COLUMNS:
        return (
            "split_key",
            False,
            "incident-held-out grouping key",
        )

    if column in METADATA_COLUMNS:
        return (
            "metadata",
            False,
            "metadata retained for audit, not default predictor",
        )

    if any(
        token in lc
        for token in POTENTIAL_LEAKAGE_TOKENS
    ):
        return (
            "review_exclude",
            False,
            "name triggers conservative potential-leakage review",
        )

    if any(
        token in lc
        for token in PROVENANCE_TOKENS
    ):
        return (
            "provenance",
            False,
            "source/run/block provenance",
        )

    if any(
        token in lc
        for token in QA_TOKENS
    ):
        return (
            "qa",
            False,
            "availability/reliability/runtime QA field",
        )

    source = source_of_column.get(
        column
    )

    if source == "landfire":

        tier = landfire_tier_map.get(
            column,
            ""
        ).lower()

        if "qa" in tier:
            return (
                "qa",
                False,
                "frozen LANDFIRE QA-only tier",
            )

        return (
            "feature_candidate",
            True,
            f"frozen LANDFIRE {tier or 'feature'} tier",
        )

    if source == "terrain":

        if column in {
            "terrain_centroid_x_m",
            "terrain_centroid_y_m",
        }:
            return (
                "metadata",
                False,
                "absolute projected location excluded by default",
            )

        return (
            "feature_candidate",
            True,
            "terrain physical feature candidate",
        )

    if source == "hrrr":

        # Date/time and object/string control fields should not enter a numeric
        # model merely because they came from HRRR.
        if (
            lc.endswith(
                "_utc"
            )
            or lc.endswith(
                "_source"
            )
        ):
            return (
                "provenance",
                False,
                "forecast timestamp/source provenance",
            )

        return (
            "feature_candidate",
            True,
            "operational HRRR feature candidate",
        )

    if source == "approach":

        if (
            lc.endswith(
                "_utc"
            )
            or lc.endswith(
                "_idx"
            )
        ):
            return (
                "provenance",
                False,
                "anchor/time index provenance",
            )

        return (
            "feature_candidate",
            True,
            "pre-anchor dynamic approach feature candidate",
        )

    return (
        "review",
        False,
        "unclassified; explicit review required",
    )


manifest_rows = []

for position, column in enumerate(
    FINAL_COLUMNS
):

    role, include_candidate, reason = (
        classify_column(
            column
        )
    )

    source = source_of_column[
        column
    ]

    manifest_rows.append({
        "position": position,
        "column": column,
        "source": source,
        "role": role,
        "candidate_predictor": bool(
            include_candidate
        ),
        "landfire_tier": (
            landfire_tier_map.get(
                column
            )
            if source
            == "landfire"
            else None
        ),
        "landfire_product": (
            landfire_product_map.get(
                column
            )
            if source
            == "landfire"
            else None
        ),
        "reason": reason,
    })


column_manifest = pd.DataFrame(
    manifest_rows
)

display(
    column_manifest.groupby(
        [
            "source",
            "role",
            "candidate_predictor",
        ],
        as_index=False,
    )
    .agg(
        columns=(
            "column",
            "size",
        )
    )
)

review_columns = column_manifest[
    column_manifest[
        "role"
    ].isin(
        [
            "review",
            "review_exclude",
        ]
    )
]

display(
    review_columns
)

column_manifest.to_csv(
    OUTPUT_ROOT /
    "MODELING_COLUMN_MANIFEST_V1.csv",
    index=False,
)

print(
    "Candidate predictors:",
    int(
        column_manifest[
            "candidate_predictor"
        ].sum()
    ),
)

print(
    "Columns requiring explicit review:",
    len(
        review_columns
    ),
)

## 8. Assemble master table with DuckDB

DuckDB performs the joins directly from Parquet so the notebook does not need
to materialize every wide source table simultaneously in pandas memory.

The assembled table is ordered by the HRRR operational base and includes only
the 304,143-unit universe.

In [ ]:
def qident(
    column,
):
    return (
        '"'
        + str(
            column
        ).replace(
            '"',
            '""'
        )
        + '"'
    )


def sql_path(
    path,
):
    return str(
        Path(
            path
        )
    ).replace(
        "'",
        "''",
    )


select_terms = []

# Core columns from HRRR.
for c in CORE_COLUMNS:
    select_terms.append(
        f'h.{qident(c)} AS {qident(c)}'
    )

# Approach unique columns.
for c in APPROACH_COLUMNS:
    select_terms.append(
        f'a.{qident(c)} AS {qident(c)}'
    )

# HRRR unique columns.
for c in HRRR_COLUMNS:
    select_terms.append(
        f'h.{qident(c)} AS {qident(c)}'
    )

# Terrain unique columns.
for c in TERRAIN_COLUMNS:
    select_terms.append(
        f't.{qident(c)} AS {qident(c)}'
    )

# LANDFIRE frozen features.
for c in LANDFIRE_COLUMNS:
    select_terms.append(
        f'l.{qident(c)} AS {qident(c)}'
    )


select_sql = ",\n        ".join(
    select_terms
)

master_tmp = MASTER_PATH.with_suffix(
    ".parquet.tmp"
)

master_tmp.unlink(
    missing_ok=True
)

con = duckdb.connect(
    database=":memory:"
)

# Keep DuckDB conservative on local-machine RAM.
con.execute(
    "PRAGMA threads=4"
)

assembly_sql = f"""
COPY (
    SELECT
        {select_sql}
    FROM read_parquet(
        '{sql_path(HRRR_PATH)}'
    ) h
    INNER JOIN read_parquet(
        '{sql_path(APPROACH_PATH)}'
    ) a
        USING (unit_uid)
    INNER JOIN read_parquet(
        '{sql_path(TERRAIN_PATH)}'
    ) t
        USING (unit_uid)
    INNER JOIN read_parquet(
        '{sql_path(LANDFIRE_PATH)}'
    ) l
        USING (unit_uid)
    ORDER BY h.unit_uid
)
TO '{sql_path(master_tmp)}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
)
"""

start = time.time()

con.execute(
    assembly_sql
)

con.close()

assert master_tmp.exists()

master_meta = pq.read_metadata(
    master_tmp
)

assert int(
    master_meta.num_rows
) == EXPECTED_OPERATIONAL_UNITS

assert list(
    master_meta.schema.names
) == FINAL_COLUMNS

master_tmp.replace(
    MASTER_PATH
)

print(
    "MASTER ASSEMBLY: WRITTEN"
)

print(
    "Rows:",
    f"{master_meta.num_rows:,}",
)

print(
    "Columns:",
    len(
        FINAL_COLUMNS
    ),
)

print(
    "Size GiB:",
    f"{MASTER_PATH.stat().st_size/1024**3:.3f}",
)

print(
    "Elapsed min:",
    f"{(time.time()-start)/60:.2f}",
)

## 9. Master-table structural and target audit

In [ ]:
master_meta = pq.read_metadata(
    MASTER_PATH
)

assert int(
    master_meta.num_rows
) == EXPECTED_OPERATIONAL_UNITS

assert list(
    master_meta.schema.names
) == FINAL_COLUMNS

master_core = pd.read_parquet(
    MASTER_PATH,
    columns=[
        "unit_uid",
        "_irwin_norm",
        "IncidentName",
        "Year",
        "_engagement_norm",
        "hold_label",
        *(
            [
                "approach_direction_reliable",
            ]
            if "approach_direction_reliable"
            in FINAL_COLUMNS
            else []
        ),
    ],
)

assert master_core[
    "unit_uid"
].is_unique

assert master_core[
    "unit_uid"
].nunique() == (
    EXPECTED_OPERATIONAL_UNITS
)

assert set(
    master_core[
        "unit_uid"
    ]
) == hrrr_ids

assert master_core[
    "_irwin_norm"
].notna().all()

assert master_core[
    "hold_label"
].notna().all()

master_target_counts = (
    master_core[
        "hold_label"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "hold_label"
    )
    .reset_index(
        name="units"
    )
)

master_target_counts[
    "pct"
] = (
    master_target_counts[
        "units"
    ]
    / len(
        master_core
    )
    * 100.0
)

incident_audit = pd.DataFrame({
    "metric": [
        "rows",
        "unique_units",
        "unique_incidents",
        "missing_target",
        "missing_incident_split_key",
    ],
    "value": [
        len(
            master_core
        ),
        master_core[
            "unit_uid"
        ].nunique(),
        master_core[
            "_irwin_norm"
        ].nunique(),
        int(
            master_core[
                "hold_label"
            ].isna().sum()
        ),
        int(
            master_core[
                "_irwin_norm"
            ].isna().sum()
        ),
    ],
})

display(
    master_target_counts
)

display(
    incident_audit
)

master_target_counts.to_csv(
    OUTPUT_ROOT /
    "MASTER_TARGET_DISTRIBUTION_V1.csv",
    index=False,
)

incident_audit.to_csv(
    OUTPUT_ROOT /
    "MASTER_INCIDENT_AUDIT_V1.csv",
    index=False,
)

print(
    "MASTER STRUCTURAL / TARGET CONTRACT: PASS"
)

## 10. Predictor missingness/type audit

No imputation is performed.

This cell inventories candidate predictors so the next modeling-design stage
can decide:

- which missingness indicators are scientifically justified;
- which features are too sparse;
- which categorical representations need encoding;
- whether core vs extended LANDFIRE tiers should be compared separately.

In [ ]:
candidate_columns = list(
    column_manifest.loc[
        column_manifest[
            "candidate_predictor"
        ],
        "column",
    ]
)

master_schema = pq.read_schema(
    MASTER_PATH
)

dtype_map = {
    field.name: str(
        field.type
    )
    for field in master_schema
}

# Efficient DuckDB aggregate missingness without loading the wide table.
con = duckdb.connect(
    database=":memory:"
)

missingness_rows = []

BATCH = 100

for start_idx in range(
    0,
    len(
        candidate_columns
    ),
    BATCH,
):

    batch = candidate_columns[
        start_idx:
        start_idx
        + BATCH
    ]

    expressions = [
        (
            f'SUM(CASE WHEN {qident(c)} IS NULL '
            f'THEN 1 ELSE 0 END) AS {qident(c)}'
        )
        for c in batch
    ]

    query = f"""
    SELECT
        {", ".join(expressions)}
    FROM read_parquet(
        '{sql_path(MASTER_PATH)}'
    )
    """

    result = con.execute(
        query
    ).fetchdf().iloc[
        0
    ]

    for c in batch:

        missing_count = int(
            result[
                c
            ]
        )

        missingness_rows.append({
            "column": c,
            "source": source_of_column[
                c
            ],
            "dtype": dtype_map.get(
                c
            ),
            "missing_count": missing_count,
            "missing_pct": (
                missing_count
                / EXPECTED_OPERATIONAL_UNITS
                * 100.0
            ),
        })


con.close()

candidate_missingness = pd.DataFrame(
    missingness_rows
)

candidate_missingness = (
    candidate_missingness.sort_values(
        [
            "missing_pct",
            "source",
            "column",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)

display(
    candidate_missingness.head(
        100
    )
)

candidate_missingness.to_csv(
    OUTPUT_ROOT /
    "CANDIDATE_PREDICTOR_MISSINGNESS_V1.csv",
    index=False,
)

print(
    "Candidate predictor columns:",
    len(
        candidate_columns
    ),
)

## 11. Final assembly checks

In [ ]:
checks = [
    {
        "check": "hrrr_exact_304143_operational_units",
        "pass": len(
            hrrr_ids
        ) == EXPECTED_OPERATIONAL_UNITS,
    },
    {
        "check": "terrain_exact_same_unit_set",
        "pass": terrain_ids == hrrr_ids,
    },
    {
        "check": "landfire_exact_same_unit_set",
        "pass": landfire_ids == hrrr_ids,
    },
    {
        "check": "approach_contains_all_operational_units",
        "pass": hrrr_ids.issubset(
            approach_ids_all
        ),
    },
    {
        "check": "frozen_cohort_contains_all_operational_units",
        "pass": hrrr_ids.issubset(
            frozen_ids_all
        ),
    },
    {
        "check": "shared_target_and_metadata_contracts_match",
        "pass": (
            True
            if not len(
                contract_audit
            )
            else bool(
                contract_audit[
                    "exact_contract_match"
                ].all()
            )
        ),
    },
    {
        "check": "approach_direction_reliability_count_frozen",
        "pass": (
            int(
                direction_valid.sum()
            )
            == EXPECTED_APPROACH_DIRECTION_RELIABLE
            and int(
                (
                    ~direction_valid
                ).sum()
            )
            == EXPECTED_APPROACH_DIRECTION_UNRELIABLE
        ),
    },
    {
        "check": "all_605_landfire_v5_fields_selected",
        "pass": len(
            LANDFIRE_FEATURES
        ) == 605
        and set(
            LANDFIRE_FEATURES
        ).issubset(
            set(
                FINAL_COLUMNS
            )
        ),
    },
    {
        "check": "final_column_names_unique",
        "pass": len(
            FINAL_COLUMNS
        ) == len(
            set(
                FINAL_COLUMNS
            )
        ),
    },
    {
        "check": "master_rows_304143",
        "pass": int(
            master_meta.num_rows
        ) == EXPECTED_OPERATIONAL_UNITS,
    },
    {
        "check": "master_exact_canonical_column_order",
        "pass": list(
            master_meta.schema.names
        ) == FINAL_COLUMNS,
    },
    {
        "check": "master_unit_uid_unique",
        "pass": master_core[
            "unit_uid"
        ].nunique()
        == EXPECTED_OPERATIONAL_UNITS,
    },
    {
        "check": "master_target_complete",
        "pass": master_core[
            "hold_label"
        ].notna().all(),
    },
    {
        "check": "master_incident_split_key_complete",
        "pass": master_core[
            "_irwin_norm"
        ].notna().all(),
    },
    {
        "check": "target_and_incident_not_candidate_predictors",
        "pass": (
            not column_manifest.loc[
                column_manifest[
                    "column"
                ].isin(
                    [
                        "hold_label",
                        "_engagement_norm",
                        "_irwin_norm",
                        "unit_uid",
                        "IncidentName",
                    ]
                ),
                "candidate_predictor",
            ].any()
        ),
    },
]

checks_df = pd.DataFrame(
    checks
)

display(
    checks_df
)

ASSEMBLY_PASS = bool(
    checks_df[
        "pass"
    ].all()
)

print()
print(
    "05A MODELING TABLE ASSEMBLY:",
    (
        "PASS / READY FOR FEATURE-SET AND INCIDENT-SPLIT DESIGN"
        if ASSEMBLY_PASS
        else "REVIEW REQUIRED"
    ),
)

checks_df.to_csv(
    OUTPUT_ROOT /
    "FINAL_ASSEMBLY_CHECKS_V1.csv",
    index=False,
)

## 12. Final report

In [ ]:
def tomd(
    df,
):
    try:
        return df.to_markdown(
            index=False
        )
    except Exception:
        return df.to_string(
            index=False
        )


role_summary = (
    column_manifest.groupby(
        [
            "source",
            "role",
            "candidate_predictor",
        ],
        as_index=False,
    )
    .agg(
        columns=(
            "column",
            "size",
        )
    )
)

missingness_summary = (
    candidate_missingness.groupby(
        "source",
        as_index=False,
    )
    .agg(
        candidate_columns=(
            "column",
            "size",
        ),
        median_missing_pct=(
            "missing_pct",
            "median",
        ),
        p90_missing_pct=(
            "missing_pct",
            lambda x: x.quantile(
                0.90
            ),
        ),
        max_missing_pct=(
            "missing_pct",
            "max",
        ),
    )
)

report = f"""# 05A Modeling Table Assembly and Feature-Provenance Audit

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Source schemas

{tomd(source_schema)}

## Operational unit-set audit

{tomd(id_audit)}

## Shared target / metadata audit

{tomd(contract_audit)}

## Target distribution

{tomd(master_target_counts)}

## Incident audit

{tomd(incident_audit)}

## Source column plan

{tomd(source_plan)}

## Column-role summary

{tomd(role_summary)}

## Candidate-predictor missingness summary

{tomd(missingness_summary)}

## Master table

Path: `{MASTER_PATH}`

Rows: `{int(master_meta.num_rows):,}`

Columns: `{len(master_meta.schema.names):,}`

Size GiB: `{MASTER_PATH.stat().st_size/1024**3:.3f}`

## Final checks

{tomd(checks_df)}

## Decision

`{"PASS / READY FOR FEATURE-SET AND INCIDENT-SPLIT DESIGN" if ASSEMBLY_PASS else "REVIEW REQUIRED"}`

05A intentionally does not create a random train/test split and does not fit a
model. The next stage should freeze predictor sets and incident-held-out split
design before any performance is measured.
"""

REPORT_PATH = (
    OUTPUT_ROOT /
    "MODELING_TABLE_ASSEMBLY_REPORT_V1.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(
    report
)

print()
print(
    "Saved:",
    REPORT_PATH,
)

## 13. Output inventory

In [ ]:
print("=" * 100)
print("05A OUTPUTS")
print("=" * 100)

for p in sorted(
    OUTPUT_ROOT.glob(
        "*"
    )
):
    if p.is_file():
        print(
            f"{p.name:<90}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

print()
print(
    "Master table:",
    MASTER_PATH,
)

print(
    "Master size:",
    f"{MASTER_PATH.stat().st_size/1024**3:.3f} GiB",
)

print()
print(
    "DONE:",
    NOTEBOOK_BUILD_ID,
)